# CryoSync: Zoho CRM -> Databricks (bronze.zoho_*)

Standalone notebook that pulls every record-bearing module from Zoho CRM and
lands it into Delta tables in the configured catalog/schema (default
`cryosync_catalog.bronze.zoho_<module>`), with an incremental watermark in
`crm_sync_log`.

Bronze table shape (self-describing, query-ready):
- `id BIGINT` (Zoho record id, merge key)
- one STRING column per Zoho field (flattened; nested values are JSON)
- `record_data STRING` (the complete original record JSON)
- `synced_at TIMESTAMP`

## Setup (do once)
1. Databricks secret scope `cryosync` with keys `zoho_client_id`, `zoho_client_secret`, `zoho_refresh_token`.
   ```
   databricks secrets create-scope --scope cryosync --initial-manage-principal users
   databricks secrets put --scope cryosync --key zoho_client_id
   databricks secrets put --scope cryosync --key zoho_client_secret
   databricks secrets put --scope cryosync --key zoho_refresh_token
   ```
2. Attach this notebook to a job/cluster that has internet egress (the driver calls Zoho's REST API) and access to the Unity Catalog catalog.
3. On the job's run parameters, set the widgets below (`catalog` / `schema` / `full_sync`).

## Usage
Run the notebook. Incremental by default (uses `Modified_Time` watermark).
Set widget `full_sync` to `true` to force a full re-pull.

In [ ]:
import json
import time
import urllib.parse
import urllib.request

# ---------------------------------------------------------------------------
# Notebook widgets + secrets
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "cryosync_catalog")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.dropdown("full_sync", "false", ["false", "true"])
dbutils.widgets.text("secret_scope", "cryosync")
dbutils.widgets.text("zoho_region", "in")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
FULL = dbutils.widgets.get("full_sync").strip().lower() == "true"
SCOPE = dbutils.widgets.get("secret_scope").strip()
REGION = dbutils.widgets.get("zoho_region").strip()

ZOHO_CLIENT_ID = dbutils.secrets.get(SCOPE, "zoho_client_id")
ZOHO_CLIENT_SECRET = dbutils.secrets.get(SCOPE, "zoho_client_secret")
ZOHO_REFRESH_TOKEN = dbutils.secrets.get(SCOPE, "zoho_refresh_token")

REGIONS = {
    "in": ("https://accounts.zoho.in", "https://www.zohoapis.in/crm/v7"),
    "us": ("https://accounts.zoho.com", "https://www.zohoapis.com/crm/v7"),
    "eu": ("https://accounts.zoho.eu", "https://www.zohoapis.eu/crm/v7"),
    "au": ("https://accounts.zoho.com.au", "https://www.zohoapis.com.au/crm/v7"),
    "jp": ("https://accounts.zoho.jp", "https://www.zohoapis.jp/crm/v7"),
    "cn": ("https://accounts.zoho.com.cn", "https://www.zohoapis.com.cn/crm/v7"),
}
ACCOUNTS_URL, API_URL = REGIONS.get(REGION, REGIONS["in"])

# Modules that are not record data (dashboards, email blobs, analytics, disabled features).
BLOCKLIST = {
    "Home", "Workqueue__s", "Activities", "Reports", "Analytics", "Social",
    "SalesInbox", "Feeds", "Documents", "Attachments", "Emails",
    "Email_Sentiment", "Email_Analytics", "Email_Template_Analytics",
    "Email_Template__s", "Email_Drafts__s", "Actions_Performed", "DealHistory",
    "Forecasts", "Forecast_Quotas", "Forecast_Items", "Forecast_Groups",
    "Locking_Information__s", "Unknown__s", "CalendarBookings__s", "Visits",
}

MAX_RETRIES = 6
FIELD_CHUNK = 50
PAGE_SIZE = 200

print(f"Catalog={CATALOG} schema={SCHEMA} full={FULL} region={REGION}")

## Zoho OAuth + REST client

In [ ]:
class ZohoClient:
    """Tiny sync Zoho CRM v7 REST client (urllib; no extra dependencies)."""

    def __init__(self):
        self.access_token = None
        self.expiry = 0.0

    def _refresh(self):
        body = urllib.parse.urlencode({
            "grant_type": "refresh_token",
            "refresh_token": ZOHO_REFRESH_TOKEN,
            "client_id": ZOHO_CLIENT_ID,
            "client_secret": ZOHO_CLIENT_SECRET,
        }).encode()
        req = urllib.request.Request(ACCOUNTS_URL + "/oauth/v2/token",
                                     data=body, method="POST")
        with urllib.request.urlopen(req, timeout=60) as r:
            tok = json.loads(r.read())
        self.access_token = tok["access_token"]
        self.expiry = time.time() + int(tok.get("expires_in", 3600)) - 120

    def _call(self, method, path, payload=None, retry=True):
        if not self.access_token or time.time() >= self.expiry:
            self._refresh()
        url = API_URL + path
        data = json.dumps(payload).encode() if payload is not None else None
        req = urllib.request.Request(url, data=data, method=method)
        req.add_header("Authorization", "Zoho-oauthtoken " + self.access_token)
        if payload is not None:
            req.add_header("Content-Type", "application/json")
        attempt = 0
        while True:
            attempt += 1
            try:
                with urllib.request.urlopen(req, timeout=120) as r:
                    raw = r.read()
                    if r.status == 204 or not raw:
                        return {}  # empty module / no records
                    return json.loads(raw)
            except urllib.error.HTTPError as e:
                code = e.code
                msg = e.read().decode("utf-8", "ignore")
                if code == 401 and retry:
                    self._refresh()
                    return self._call(method, path, payload, retry=False)
                if code == 429:
                    wait = 5
                    try:
                        wait = float(e.headers.get("Retry-After") or 5)
                    except (TypeError, ValueError):
                        pass
                    time.sleep(min(wait, 60))
                elif code >= 500 and retry:
                    time.sleep(5)
                elif code == 400 and "LIMIT_EXCEEDED" in msg:
                    raise RuntimeError(f"Zoho {path} -> 400 {msg}")
                else:
                    raise RuntimeError(f"Zoho {path} -> {code}: {msg}")
            except urllib.error.URLError:
                time.sleep(5)
            except (json.JSONDecodeError, UnicodeDecodeError):
                time.sleep(5)
            if attempt > MAX_RETRIES:
                raise RuntimeError(f"Zoho {path} still failing after retries")
            time.sleep(2 * attempt)

    def modules(self):
        data = self._call("GET", "/settings/modules?include=1")
        out = []
        for m in data.get("modules", []):
            api = m.get("api_name") or m.get("module_name")
            if api and api not in BLOCKLIST:
                out.append(api)
        return sorted(out)

    def fields(self, module):
        data = self._call("GET", "/settings/fields?module=" + urllib.parse.quote(module))
        return [f.get("api_name") for f in data.get("fields", []) if f.get("api_name")]

    def records(self, module, fields, since=None):
        """Yield complete records, handling the 50-field cap and pagination."""
        chunks = [fields[i:i + FIELD_CHUNK] for i in range(0, len(fields), FIELD_CHUNK)]
        if not chunks:
            chunks = [["All"]]

        def fetch(chunk, page):
            q = {"per_page": PAGE_SIZE, "page": page, "fields": ",".join(chunk)}
            if since:
                q["criteria"] = f"(Modified_Time:greater_than:{since})"
            data = self._call("GET", f"/{module}?{urllib.parse.urlencode(q)}")
            info = data.get("info") or {}
            return data.get("data") or [], info.get("more_records", False)

        page = 1
        while True:
            merged = {}
            any_data = False
            any_more = False
            for chunk in chunks:
                rows, more = fetch(chunk, page)
                any_data = any_data or bool(rows)
                any_more = any_more or more
                for r in rows:
                    rid = r.get("id")
                    if rid not in merged:
                        merged[rid] = {}
                    merged[rid].update(r)
            if not any_data:
                return
            for rid in merged:
                yield merged[rid]
            if not any_more:
                return
            page += 1


zoho = ZohoClient()

## Databricks Delta writes (MERGE on id)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               TimestampType)


def sanitize(value):
    if value is None:
        return None
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False, default=str)
    if isinstance(value, bool):
        return "true" if value else "false"
    return str(value)


def table_name(module):
    return f"{CATALOG}.{SCHEMA}.zoho_{module.lower()}"


def ensure_table(table, cols):
    """Create if missing; add any new flattened columns as STRING."""
    col_spec = ", ".join(["id BIGINT"] + [f"`{c}` STRING" for c in cols] +
                         ["record_data STRING", "synced_at TIMESTAMP"])
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} ({col_spec}) USING DELTA")
    existing = {r["col_name"].lower() for r in
                spark.sql(f"DESCRIBE TABLE {table}").collect()}
    for c in cols:
        if c.lower() not in existing and c.lower() != "record_data":
            spark.sql(f"ALTER TABLE {table} ADD COLUMN `{c}` STRING")


def upsert(table, payload_cols, rows):
    """MERGE a batch of records into the Delta table on id."""
    if not rows:
        return 0
    schema = StructType(
        [StructField(c, LongType() if c == "id" else StringType(), True)
         for c in payload_cols]
    )
    df = spark.createDataFrame(rows, schema=schema)
    df.createOrReplaceTempView("zoho_src")
    upd = ", ".join(f"t.`{c}` = s.`{c}`" for c in payload_cols if c != "id")
    ins_cols = ", ".join(f"`{c}`" for c in payload_cols)
    ins_vals = ", ".join(f"s.`{c}`" for c in payload_cols)
    spark.sql(
        f"MERGE INTO {table} t USING zoho_src s ON t.id = s.id "
        f"WHEN MATCHED THEN UPDATE SET {upd} "
        f"WHEN NOT MATCHED THEN INSERT ({ins_cols}) VALUES ({ins_vals})"
    )
    return len(rows)


def watermark(module):
    if not spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.crm_sync_log"):
        return None
    rows = spark.sql(
        f"SELECT last_modified_time FROM {CATALOG}.{SCHEMA}.crm_sync_log "
        f"WHERE module = '{module.replace(chr(39), chr(39)+chr(39))}'"
    ).collect()
    return rows[0][0] if rows and rows[0][0] else None


def set_watermark(module, modified_time):
    spark.sql(
        f"MERGE INTO {CATALOG}.{SCHEMA}.crm_sync_log t "
        f"USING (SELECT '{module.replace(chr(39), chr(39)+chr(39))}' AS module, "
        f"       '{modified_time.replace(chr(39), chr(39)+chr(39))}' AS last_modified_time) s "
        f"ON t.module = s.module "
        f"WHEN MATCHED THEN UPDATE SET last_modified_time = s.last_modified_time "
        f"WHEN NOT MATCHED THEN INSERT (module, last_modified_time) VALUES (s.module, s.last_modified_time)"
    )


spark.sql(f"CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.crm_sync_log ("
          "module STRING NOT NULL, last_modified_time STRING, last_synced TIMESTAMP) USING DELTA")

## Sync one module

In [ ]:
def sync_module(module):
    t0 = time.time()
    table = table_name(module)
    ofields = zoho.fields(module)
    if "id" in ofields:
        ofields.remove("id")
    if "record_data" in ofields:
        ofields.remove("record_data")
    cols = list(ofields)

    ensure_table(table, cols)
    since = None if FULL else watermark(module)

    payload_cols = ["id"] + cols + ["record_data", "synced_at"]
    synced_at = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(t0))

    rows = []
    total = 0
    upserted = 0
    max_mod = since
    for rec in zoho.records(module, cols, since=since):
        rid = rec.get("id")
        if rid is None:
            continue
        flat = {"id": int(rid)}
        for c in cols:
            flat[c] = sanitize(rec.get(c)) if c in rec else None
        flat["record_data"] = json.dumps(rec, ensure_ascii=False, default=str)
        flat["synced_at"] = synced_at
        rows.append(flat)
        mod = rec.get("Modified_Time")
        if mod and (max_mod is None or mod > max_mod):
            max_mod = mod
        total += 1
        if len(rows) >= PAGE_SIZE:
            upserted += upsert(table, payload_cols, rows)
            rows = []
    if rows:
        upserted += upsert(table, payload_cols, rows)

    if max_mod:
        set_watermark(module, max_mod)

    mode = "incremental" if since else "full"
    print(f"[{module}] {mode}: pulled={total} upserted={upserted} cols={len(cols)} "
          f"in {round(time.time() - t0, 1)}s")
    return total

## Run all modules

In [ ]:
modules = zoho.modules()
print(f"Syncing {len(modules)} modules ...")
grand_total = 0
failed = []
for i, m in enumerate(modules):
    try:
        grand_total += sync_module(m)
    except Exception as e:
        failed.append((m, str(e)))
        print(f"[{m}] FAILED: {e}")
    if i < len(modules) - 1:
        time.sleep(1.5)

print(f"DONE: {len(modules)} modules / {grand_total} records, failed={len(failed)}")
if failed:
    print("Failed modules:", failed)